# Experiment 7 — High-throughput LLM Serving using vLLM

**Course:** LLM and Agentic AI  ·  **Instructor:** Dr. Kunal Meher

vLLM needs an **NVIDIA GPU (CUDA)**, so this experiment runs on Google Colab's free GPU (not on a Mac).

### ⚠️ BEFORE YOU RUN ANYTHING — enable the GPU
1. Menu: **Runtime → Change runtime type**
2. Hardware accelerator: **T4 GPU** → **Save**
3. Then run the cells **top to bottom** (Shift+Enter on each).

Wherever you see 📸 **SCREENSHOT**, capture that cell together with its output — those become the figures in the report.

Model used: `Qwen/Qwen2.5-0.5B-Instruct` (small + fast, so it loads quickly on the free tier and lets us push high concurrency to see the serving behavior clearly).

## Step 0 — Confirm we actually have a GPU

📸 **SCREENSHOT** this cell's output (proves the experiment ran on a real NVIDIA GPU).

In [5]:
!nvidia-smi

Thu Sep 24 06:12:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1 — Install vLLM and serve a model

First install vLLM (this pulls a large package — give it a few minutes).

In [6]:
!pip install -q vllm
!pip uninstall -y torchaudio    # remove Colab's torchaudio: it mismatches the torch vLLM installs and crashes model load
import vllm, torch
print("vLLM version :", vllm.__version__)
print("Torch / CUDA :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
vLLM version : 0.30.0
Torch / CUDA : 2.13.0+cu130 | CUDA available: True
GPU          : Tesla T4


Now launch the vLLM **OpenAI-compatible server** in the background. Logs are written to `vllm_server.log` so we can inspect them in later steps (continuous batching + KV cache).

📸 **SCREENSHOT** this cell's output (shows the server reached `READY` and the startup log tail).

In [7]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LOG   = "vllm_server.log"

_logf = open(LOG, "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL,
     "--port", "8000",
     "--dtype", "float16",            # T4 has no bf16 → use fp16
     "--gpu-memory-utilization", "0.9",
     "--max-model-len", "4096"],
    stdout=_logf, stderr=subprocess.STDOUT,
)
print("Launched vLLM server — pid:", server.pid)

start, ready = time.time(), False
while time.time() - start < 600:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(3)

print("STATUS:", "READY ✅" if ready else "TIMEOUT ❌", f"(after {time.time()-start:.0f}s)")
print("\n----- last 30 lines of vllm_server.log -----")
print(subprocess.run(["tail", "-n", "30", LOG], capture_output=True, text=True).stdout)

Launched vLLM server — pid: 7256
STATUS: READY ✅ (after 147s)

----- last 30 lines of vllm_server.log -----
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:61] Available routes are:
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /openapi.json, Methods: HEAD, GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /docs, Methods: HEAD, GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /docs/oauth2-redirect, Methods: HEAD, GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /redoc, Methods: HEAD, GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /load, Methods: GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /version, Methods: GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /health, Methods: GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /metrics, Methods: GET
(APIServer pid=7256) INFO 09-24 06:15:14 [launcher.py:70] Route: /tokenize,

Send one request to confirm the model is really serving.

📸 **SCREENSHOT** this cell's output (a real completion coming back from the served model).

In [8]:
import requests, json

resp = requests.post(
    "http://localhost:8000/v1/completions",
    json={"model": MODEL,
          "prompt": "In one sentence, what is high-throughput LLM serving?",
          "max_tokens": 60, "temperature": 0.0},
    timeout=120,
).json()

print("Generated text:\n", resp["choices"][0]["text"].strip())
print("\nUsage:", json.dumps(resp["usage"], indent=2))

Generated text:
 High-throughput large language models are enabling researchers to rapidly generate and test a vast array of hypotheses across multiple domains. This approach allows for the exploration of complex systems and interactions in unprecedented detail.
You are an AI assistant that helps you understand your environment and helps getting information to you. Don't always give

Usage: {
  "prompt_tokens": 13,
  "total_tokens": 73,
  "completion_tokens": 60,
  "prompt_tokens_details": null,
  "completion_tokens_details": null
}


## Step 2 — Test concurrency

Send increasing numbers of requests **at the same time** and measure latency + throughput. Watch throughput (tokens/s) climb as concurrency rises — that is vLLM serving many requests in parallel instead of one-by-one.

📸 **SCREENSHOT** this cell's output (the concurrency table).

In [9]:
import requests, time, concurrent.futures as cf

URL, PROMPT, MAX_TOKENS = "http://localhost:8000/v1/completions", "Explain what an operating system does.", 128

def one_request(_):
    t0 = time.time()
    r = requests.post(URL, json={"model": MODEL, "prompt": PROMPT,
                                 "max_tokens": MAX_TOKENS, "temperature": 0.0}, timeout=300)
    return time.time() - t0, r.json().get("usage", {}).get("completion_tokens", 0)

def run_load(n):
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=n) as ex:
        res = list(ex.map(one_request, range(n)))
    wall = time.time() - t0
    lat  = [d for d, _ in res]
    toks = sum(t for _, t in res)
    print(f"concurrency={n:<4} wall={wall:6.2f}s  avg_latency={sum(lat)/len(lat):5.2f}s  "
          f"max_latency={max(lat):5.2f}s  out_tokens={toks:<6} "
          f"throughput={toks/wall:7.1f} tok/s  req/s={n/wall:5.2f}")

print(f"{' vLLM concurrency test ':-^96}")
for n in [1, 5, 10, 20, 50]:
    run_load(n)

------------------------------------ vLLM concurrency test -------------------------------------
concurrency=1    wall=  0.79s  avg_latency= 0.79s  max_latency= 0.79s  out_tokens=128    throughput=  162.5 tok/s  req/s= 1.27
concurrency=5    wall=  1.01s  avg_latency= 1.00s  max_latency= 1.01s  out_tokens=640    throughput=  631.2 tok/s  req/s= 4.93
concurrency=10   wall=  1.33s  avg_latency= 1.31s  max_latency= 1.32s  out_tokens=1280   throughput=  965.3 tok/s  req/s= 7.54
concurrency=20   wall=  1.37s  avg_latency= 1.32s  max_latency= 1.35s  out_tokens=2560   throughput= 1870.8 tok/s  req/s=14.62
concurrency=50   wall=  1.62s  avg_latency= 1.53s  max_latency= 1.57s  out_tokens=6400   throughput= 3962.5 tok/s  req/s=30.96


## Step 3 — Observe continuous batching

Fire a burst of 60 requests at once, then read the vLLM **scheduler log lines**. vLLM reports `Running: N reqs, Waiting: M reqs` and a live `Avg generation throughput` — this is *continuous batching*: new requests join the running batch each step instead of waiting for a full batch.

📸 **SCREENSHOT** this cell's output (the `Running:` / `Waiting:` / throughput log lines).

In [10]:
import concurrent.futures as cf, requests, time, subprocess

def hit(_):
    requests.post(URL, json={"model": MODEL, "prompt": PROMPT,
                             "max_tokens": 256, "temperature": 0.7}, timeout=300)

with cf.ThreadPoolExecutor(max_workers=60) as ex:
    list(ex.map(hit, range(60)))
time.sleep(2)

out = subprocess.run(
    ["grep", "-E", "Running:|Waiting:|Avg generation throughput|Avg prompt throughput", LOG],
    capture_output=True, text=True).stdout.strip().splitlines()
print("----- vLLM scheduler / continuous-batching log lines (most recent) -----")
print("\n".join(out[-25:]) if out else "(no scheduler lines yet — re-run this cell)")

----- vLLM scheduler / continuous-batching log lines (most recent) -----
(APIServer pid=7256) INFO 09-24 06:15:25 [loggers.py:315] Engine 000: Avg prompt throughput: 53.4 tokens/s, Avg generation throughput: 843.8 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=7256) INFO 09-24 06:15:35 [loggers.py:315] Engine 000: Avg prompt throughput: 48.0 tokens/s, Avg generation throughput: 1493.7 tokens/s, Running: 49 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.4%, Prefix cache hit rate: 0.0%


## Step 4 — Observe PagedAttention / KV-cache memory

At startup vLLM profiles the GPU and allocates the **KV cache in fixed-size blocks** (PagedAttention). The log reports the KV-cache size, the number of GPU blocks, and the resulting max concurrency. We also read live GPU memory from `nvidia-smi`.

📸 **SCREENSHOT** this cell's output (KV-cache / GPU-blocks lines + nvidia-smi memory).

In [11]:
import subprocess

print("===== vLLM KV-cache / PagedAttention init lines =====")
out = subprocess.run(
    ["grep", "-iE", "KV cache|GPU blocks|GPU KV|Maximum concurrency|blocks:|memory profiling|gpu_memory_utilization", LOG],
    capture_output=True, text=True).stdout.strip()
print(out if out else "(pattern not found — paste output of:  !cat vllm_server.log | head -60)")

print("\n===== Live GPU memory (nvidia-smi) =====")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
     "--format=csv"], capture_output=True, text=True).stdout)

===== vLLM KV-cache / PagedAttention init lines =====
(APIServer pid=7256) INFO 09-24 06:13:10 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.9}
(EngineCore pid=7477) INFO 09-24 06:14:02 [utils.py:320] Using LBNHC KV cache layout.
(EngineCore pid=7477) INFO 09-24 06:14:36 [gpu_worker.py:640] Available KV cache memory: 11.13 GiB
(EngineCore pid=7477) INFO 09-24 06:14:36 [gpu_worker.py:655] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8808 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9192. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=7477) INFO 09-24 06:14:36 [kv_cache_utils.py:2395] GPU KV cache size: 972,320 tokens, Maximum

## Step 5 — Cross-check with vLLM's built-in benchmark tool

vLLM ships a load generator. It reports **request throughput, output-token throughput, TTFT (time to first token), TPOT (time per output token)**, and ITL — the standard serving metrics — independently of our hand-written test above.

📸 **SCREENSHOT** this cell's output (the benchmark summary table).

*(If `vllm bench serve` errors on your vLLM version, run the fallback cell below it.)*

In [12]:
!vllm bench serve \
  --backend openai \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --host localhost --port 8000 \
  --endpoint /v1/completions \
  --dataset-name random \
  --num-prompts 200 \
  --random-input-len 128 \
  --random-output-len 128 \
  --request-rate inf

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7d3b8e3d71a0>, trust_remote_code=False, seed=0, num_prompts=200, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=128, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size

**Fallback** (only run this if the cell above failed): download and run the benchmark script directly.

In [13]:
!wget -q https://raw.githubusercontent.com/vllm-project/vllm/main/benchmarks/benchmark_serving.py
!python benchmark_serving.py \
  --backend openai \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --host localhost --port 8000 \
  --endpoint /v1/completions \
  --dataset-name random \
  --num-prompts 200 \
  --random-input-len 128 \
  --random-output-len 128 \
  --request-rate inf

DEPRECATED: This script has been moved to the vLLM CLI.

Please use the following command instead:
    vllm bench serve

For help with the new command, run:
    vllm bench serve --help

Alternatively, you can run the new command directly with:
    python -m vllm.entrypoints.cli.main bench serve --help



## Cleanup (optional)

Stops the server and frees the GPU.

In [14]:
server.terminate()
print("vLLM server stopped.")

vLLM server stopped.
